In [1]:
import requests
import numpy as np
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords
import re
from collections import defaultdict
index=0
from nltk.corpus import wordnet as wn
english_stemmer = PorterStemmer()
arabic_stemmer = ISRIStemmer()
lemmatizer = WordNetLemmatizer()
english_stopwords = set(stopwords.words('english'))
english =set(stopwords.words('english'))
english=english.difference({"and","or","not"})
arabic_stopwords = set(stopwords.words('arabic'))
def is_arabic(word):
    return any('\u0600' <= c <= '\u06FF' for c in word)

In [2]:
def process_text(text):
    tokens = word_tokenize(text.lower())
    processed = []
    for token in tokens:
        if not re.match(r'^[a-zA-Z\u0600-\u06FF]+$', token):
            continue
        if is_arabic(token):
            if token in arabic_stopwords:
                continue
            stemmed = arabic_stemmer.stem(token)
        else:
            if token in english_stopwords:
                continue
            stemmed = english_stemmer.stem(token)
            stemmed = lemmatizer.lemmatize(stemmed)
        #stemmed = re.sub(r'[^a-zA-Z\u0600-\u06FF]', '', stemmed)
        if stemmed:
            processed.append(stemmed)
    return processed

In [3]:
def process_text_user(text):
    tokens = word_tokenize(text.lower())
    processed = []
    for token in tokens:
        if not re.match(r'^[a-zA-Z\u0600-\u06FF]+$', token):
            continue
        if is_arabic(token):
            if token in arabic_stopwords:
                continue
            stemmed = arabic_stemmer.stem(token)
        else:
            if token in english:
                continue
            stemmed = english_stemmer.stem(token)
            stemmed = lemmatizer.lemmatize(stemmed)
        #stemmed = re.sub(r'[^a-zA-Z\u0600-\u06FF]', '', stemmed)
        if stemmed:
            processed.append(stemmed)
    return processed

In [4]:
alltext = dict()
def buildIndex(docs,link):
    global alltext
    index= dict()
    for i in range(len(docs)):
        text = [] 
        for p in docs[i]:
            pp=p.get_text()
            terms = process_text(pp)
            text.append(terms)
            for t in terms :
                t = t.lower()
                if t in index.keys():
                    l = index.get(t)
                else:
                    l = []
                if link[i] not in l:
                    l.append(link[i])
                index.update({t : l })
        alltext.update({link[i]:text})
    return index

In [5]:
def search(query):
    i=0
    max_similarity=-1
    if query in index.keys():
        #check(query)
        return index[query]
    else:
        for w in wn.synsets(query):
            for word in index.keys():
                if(len(wn.synsets(word))>0):
                    similarity = w.wup_similarity(wn.synsets(word)[0])
                    if similarity is not None and similarity > max_similarity:
                        max_similarity = similarity
                        most_similar_word = word                 
        if   max_similarity>-1:
            print(f"This word '{query}' not found in website the similar word:'{most_similar_word}' parcent:{max_similarity}")
            #check(most_similar_word)
            return index[most_similar_word]
        else:
            print("this word not found in website")
        

In [6]:
def andSearch(t1 , t2): 
    print("and:")
    res = list()
    if type(t1) is not list:
        res1 = search(t1.lower())
    else:
        res1 = t1
    print(f"{t1} = {res1}")
    res2 = search(t2)
    print(f"{t2} = {res2}")
    if(res1 is None or res2 is None):
        return res
    for i in res1:
        for j in res2:
            if(i == j):
                res.append(i)  
    print(f"res={res}")
    print("--------")
    return res

In [7]:
def orSearch(t1 , t2):
    print("or")
    res = list()
    if type(t1) is not list:
        res1 = search(t1.lower())
    else:
        res1 = t1
    print(f"{t1} = {res1}")
    res2 = search(t2.lower())
    print(f"{t2} = {res2}")
    if(res1 is None):
        return res2
    if(res2 is None):
        return res1
    res = set(res1+res2)
    print(f"res={res}")
    print("--------")
    return res

In [8]:
def notSearch(t1,t2):
    print("not:")
    res = list()
    if type(t1) is not list:
        res1 = search(t1.lower())
    else:
        res1 = t1
    res2 = search(t2)
    if(res1 is None):
        return res
    if(res2 is None):
        return res1
    print(f"{t1} = {res1}")
    print(f"{t2} = {res2}")
    for i in res1:
            if i not in res2:
                res.append(i)
    print(f"res={res}")
    print("--------")
    return res

In [9]:
import requests
import re
from bs4 import BeautifulSoup
count_document=0
class crawler:
    def __init__(self):
        self.to_visit = list()
        self.visited = set()

    def fetch(self, url):
        print('now fetching.. ' , url)
        
        res = requests.get(url).content

        return res

    def get_current_url(self):
        res = self.to_visit.pop()

        while res in self.visited:
            #print('already visited',res)
            res = self.to_visit.pop()

        return res

    def get_links(self , content ):
        urls = re.findall('<a href="([^"]+)">', str(content))
        #print('urls are', urls)
        for url in urls:
            pattern = re.compile('http?')
            if pattern.match(url):
                self.to_visit.append(url)

    def  crawl(self, url, depth):
        depth=int(depth)
        global index
        self.to_visit.append(url)
        l=[]
        link=[]
        while len(self.visited) <= depth:
            current_url = self.get_current_url()
            link.append(current_url)
            content = self.fetch(current_url)
            omar=BeautifulSoup(content,'html.parser')
            p1=omar.find_all('p')
            l.append(p1)
            self.visited.add(current_url)
            self.get_links(content)
        #print(self.to_visit)
       # print(self.visited)
        
        global count_document
        count_document = len(link)
        index=buildIndex(l,link)

In [10]:
def compute_idf(word):  
    counter =  len(index.get(word, []))
    return np.log(count_document)/ (counter + 1)
    
def compute_tf(w,doc): 
    words_count=0
    word_count=0
    target= alltext[doc]
    for words in target:
        for word in words:
            if(w in word):
                word_count +=1
            words_count +=1   
    return word_count/ words_count
def check():
    tf_idf= dict() 
    for key in index.keys():
        idf=compute_idf(key)
        for url in index[key]:
            tf=compute_tf(key,url)
            tf_idf.update({url:idf*tf})
            sorted_tf_idf = sorted(tf_idf.items(), key=lambda x: x[1], reverse=True)
            sorted_tf_idf_dict = dict(sorted_tf_idf)
        urls = list(sorted_tf_idf_dict.keys())
        index.update({key : urls })
        #print(f"The Rnacing of '{word}' is: {sorted_tf_idf_dict}")
        tf_idf.clear()
        sorted_tf_idf_dict.clear()


In [11]:
def inp_sen():
    sentence = input("Pleas, Enter the sentence")
    res = list()
    word = list()
    word = process_text_user(sentence)
    words = [w for w in word if w.lower() not in ['i', 'for', 'is']]
    print(words)
    i = 0
    j = len(words)
    while i < j:
        if i <= len(words) and words[i] in "and":
            if len(res) == 0:
                res1 = andSearch(words[i - 1], words[i + 1])
                res.append(res1)
            else:
                print(res[0])
                res1 = andSearch(res.pop(0), words[i + 1])
                res.append(res1)

        elif i <= len(words) and words[i] in "or":
            if len(res) == 0:
                res1 = orSearch(words[i - 1], words[i + 1])
                res.append(res1)
            else:
                res1 = orSearch(res.pop(0), words[i + 1])
                res.append(res1)

        elif i <= len(words) and words[i] in "not":
            if len(res) == 0:
                res1 = notSearch(words[i - 1], words[i + 1])
                res.append(res1)
            else:
                res1 = notSearch(res.pop(0), words[i + 1])
                res.append(res1)

        elif j == 1:
            res1 = search(words[0])
            res.append(res1)

        elif i > 0 and i <= j - 1 and words[i - 1] not in ["and", "or", "not"]:
            if len(res) == 0:
                res1 = andSearch(words[i - 1], words[i])
                res.append(res1)
            else:
                res1 = andSearch(res.pop(0), words[i])
                res.append(res1)
        i = 1 + i

    # تعديل طريقة عرض النتيجة
    if res and isinstance(res[0], list):
        print("النتائج:")
        for result in res[0]:
            print(f"- {result}")
        print("النتائج على شكل لست")
    else:
        print("لا توجد نتائج.")
    
    return res


In [12]:
counter=1
link=input("Pleas, Enter the Link")
depth=input("Pleas, Enter the Depth of Link")
c = crawler()
c.crawl(link,depth)
print("The number of Document in website is:")
print(count_document)
print("The number of words in link is:")
print(len(index))
print("The index of website is:")
check()
print(index)

Pleas, Enter the Link https://ar.m.wikipedia.org/wiki/%D8%B0%D9%83%D8%A7%D8%A1_%D8%A7%D8%B5%D8%B7%D9%86%D8%A7%D8%B9%D9%8A 
Pleas, Enter the Depth of Link 5


now fetching..  https://ar.m.wikipedia.org/wiki/%D8%B0%D9%83%D8%A7%D8%A1_%D8%A7%D8%B5%D8%B7%D9%86%D8%A7%D8%B9%D9%8A 
now fetching..  https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use
now fetching..  https://stats.wikimedia.org/#/foundation.wikimedia.org
now fetching..  https://developer.wikimedia.org
now fetching..  https://www.mediawiki.org/wiki/Special:MyLanguage/Developer_Portal/Contribute
now fetching..  https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement
The number of Document in website is:
6
The number of words in link is:
1940
The index of website is:
{'ذكء': ['https://ar.m.wikipedia.org/wiki/%D8%B0%D9%83%D8%A7%D8%A1_%D8%A7%D8%B5%D8%B7%D9%86%D8%A7%D8%B9%D9%8A '], 'صطناع': ['https://ar.m.wikipedia.org/wiki/%D8%B0%D9%83%D8%A7%D8%A1_%D8%A7%D8%B5%D8%B7%D9%86%D8%A7%D8%B9%D9%8A '], 'صنع': ['https://ar.m.wikipedia.org/wiki/%D8%B0%D9%83%D8%A7%D8%A1_%D8%A7%D8%B5%D8%B7%D9%86%D8%A7%D8%B9%D9%8A '], 'انجليزية': ['https://ar.m.wikiped

In [ ]:
while(counter==1):
   x=inp_sen()
   print(x)
   print("--------------------------(-_-)-----------------------------")
   counter=int(input("Pleas, Enter 1 to replay or 0 to stop"))

Pleas, Enter the sentence may


['may']
النتائج:
- https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement
- https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use
النتائج على شكل لست
[['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']]
--------------------------(-_-)-----------------------------


Pleas, Enter 1 to replay or 0 to stop 1
Pleas, Enter the sentence ttt


['ttt']
this word not found in website
لا توجد نتائج.
[None]
--------------------------(-_-)-----------------------------


Pleas, Enter 1 to replay or 0 to stop 1
Pleas, Enter the sentence ttt and active


...

['may', 'and', 'activ']
and:
may = ['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
activ = ['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
res=['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
--------
[['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']]
--------------------------(-_-)-----------------------------


Pleas, Enter 1 to replay or 0 to stop 1
Pleas, Enter the sentence active not may


['activ', 'not', 'may']
not:
activ = ['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
may = ['https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Cookie_statement', 'https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
res=[]
--------
[[]]
--------------------------(-_-)-----------------------------


Pleas, Enter 1 to replay or 0 to stop 1
Pleas, Enter the sentence i love ai


['love', 'ai']
and:
This word 'love' not found in website the similar word:'fear' parcent:0.8571428571428571
love = ['https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
This word 'ai' not found in website the similar word:'human' parcent:0.7857142857142857
ai = ['https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
res=['https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']
--------
[['https://foundation.m.wikimedia.org/wiki/Special:MyLanguage/Policy:Terms_of_Use']]
--------------------------(-_-)-----------------------------


In [ ]:
print(alltext)